In [2]:
import pandas as pd
df = pd.read_csv('for_export_philippine_elections.csv')
df.head()

authors = pd.read_csv('well_known_authors_philippine_elections.csv')

In [3]:
import re

def filter_text(text):
    new_text = re.sub(r'https?://\S+|www\.\S+', r'', str(text))
    new_text = re.sub(r'\@\S+', r'', str(new_text))
    new_text = re.sub(r'[^\x00-\x7f]', r'', str(new_text))
    new_text = re.sub(r'[^\w\s\@\#]', r' ', str(new_text))
    new_text = new_text.lower()
    return new_text

df['text_filtered'] = df['text'].apply(filter_text)

df.head()

,pseudo_id,text,retweetCount,replyCount,likeCount,quoteCount,viewCount,bookmarkCount,createdAt,lang,isReply,pseudo_conversationId,pseudo_inReplyToUsername,pseudo_author_userName,author_isBlueVerified,text_filtered
0,224179509746025,not a fan of Brent pero ano ba 'yang Big Winne...,0,0,0,0,5,0,2025-05-31 16:22:41+00:00,tl,False,224179509746025,NaN,733727272962306,False,not a fan of brent pero ano ba yang big winne...
1,998635197269390,wala talagang aasahan sa mga botante JUSKOPO,0,0,0,0,7,0,2025-05-31 16:21:56+00:00,tl,False,998635197269390,NaN,924044017517801,False,wala talagang aasahan sa mga botante juskopo
2,954085072967672,@975452594351818 Magiging presidente para guma...,0,0,0,0,0,0,2025-05-31 16:21:03+00:00,tl,True,725047262887831,9.754526e+14,685540093851332,False,magiging presidente para gumanti hindi para ...
3,90429221872722,Bumoto kahit walang idea kung ano to. Promise ...,0,0,0,0,2,0,2025-05-31 16:20:20+00:00,tl,False,90429221872722,NaN,897590766311451,False,bumoto kahit walang idea kung ano to promise ...
4,503936424634617,alam mo talagang wikla duo ang bet ng CVs kase...,0,0,0,0,1,0,2025-05-31 16:19:10+00:00,tl,False,503936424634617,NaN,887555520780376,False,alam mo talagang wikla duo ang bet ng cvs kase...


In [34]:
import stopwordsiso as stopwords

word_tokens_list = [tweet.split() for tweet in df['text_filtered']]

# stop words for english and tagalog
stop_words = stopwords.stopwords(["en", "tl"])
stop_words.update(['yan', 'yang', 'naman', 'lang', 'yung', 'di', 'niyo', 'nyo', 'sana', 'na', 'nga', 'kasi', 'po', 'natin', 'mag',
                    'kay', 'talaga', 'nalang', 'jan', 'nung', 'parang', 'pala', 'ung', 'kayong', 'mas', 'rin', 'din', 'kayo',
                   'pag', 'rin', 'daw', 'raw', 'yun', 'nag', 'sya', 'nya', 'nang', 'upang', 'ayon', 'upang', 'nitong', 'pang', 
                   'inyo', 'dyan', 'lng', 'ngayong', 'wag', 'baka', 'pls', 'para', 'nasa', 'muna', 'sige', 'puro', 'lalo', 'mong',
                   'ibang', 'anong'])
stop_words.update(['binoto', 'botohan', 'boto', 'vote', 'iboto', 'halalan', 'iboboto', 'bumoto', '#halalan2025', 'eleksyon2025',
                   'iboboto', 'boto', 'vote', 'votes', 'elections', 'election', 'politics', '2025', 'philippine', 'voting',
                   '1', '2', 'pilipinas', 'kc', 'nating', 'ninyo'])
filtered_tokens = []
temp_filtered_tokens = []
for word_tokens in word_tokens_list:
    temp_filtered_tokens.append([token for token in word_tokens if (token.lower() not in stop_words)])

author_usernames = dict(zip(
    authors['obfuscated_userName'], 
    authors['author_userName']
))

print(author_usernames)

for word_tokens in temp_filtered_tokens:
    filtered_tokens.append([
        author_usernames[token] if (author_usernames.get(token))
        else token for token in word_tokens
    ])

# for x in temp_filtered_tokens:
#     print(x)
print("------------------")
for x in filtered_tokens:
    print(x)

{'@447129996589185': '__nisanjo', '@502848624539866': '_djchacha', '@894831624795349': '_GaiaPoly', '@344694405381762': 'ABSCBN', '@403780272519270': 'ABSCBNNews', '@34859533045326': 'ABSCBNpr', '@456793039343990': 'AFP', '@876575320724228': 'AHOF__GRANDMA', '@347280075511207': 'akazukinchan2x', '@11503056031366': 'akoposimarcelo', '@526730790366201': 'AltStarMagic', '@467161425997003': 'alvinelchico', '@802585864361733': 'amaxtriton', '@208287835608527': 'ANCALERTS', '@936960220706872': 'apangilinan', '@70047054001669': 'Arshiethromycin', '@712192195397021': 'AskPerplexity', '@575306104619656': 'ateneodemanilau', '@328126839578840': 'attyharryroque', '@894416324110752': 'AttyLeiladeLima', '@730798648352396': 'AttyMatula', '@951392445303700': 'bamaquino', '@768795647400867': 'BangkoSentral', '@33643637499439': 'barrygutierrez3', '@326683475544033': 'bayanmo', '@942501146785377': 'BbMaharlika', '@979339868045075': 'beabinene', '@69469140131426': 'bgyo_ph', '@607504780903547': 'BiniJhoan

In [61]:
from gensim.corpora.dictionary import Dictionary
from gensim.models.ldamodel import LdaModel

dictionary = Dictionary(filtered_tokens)
corpus = [dictionary.doc2bow(text) for text in filtered_tokens]

lda = LdaModel(
    corpus=corpus, 
    id2word=dictionary, 
    num_topics=8
)
lda.print_topics()

[(0,
  '0.019*"tama" + 0.015*"bam" + 0.014*"kiko" + 0.014*"period" + 0.011*"aquino" + 0.011*"villar" + 0.011*"ayuda" + 0.009*"awa" + 0.008*"guys" + 0.007*"bini"'),
 (1,
  '0.015*"wala" + 0.012*"tapos" + 0.012*"alam" + 0.010*"hahaha" + 0.009*"#sb19" + 0.008*"time" + 0.008*"pera" + 0.007*"ganyan" + 0.007*"haha" + 0.006*"loob"'),
 (2,
  '0.016*"bayan" + 0.015*"kandidato" + 0.013*"china" + 0.011*"senado" + 0.009*"darating" + 0.009*"senador" + 0.009*"bansa" + 0.008*"pilipino" + 0.008*"sara" + 0.008*"duterte"'),
 (3,
  '0.051*"comelec" + 0.025*"commission" + 0.010*"national" + 0.009*"campaign" + 0.007*"midterm" + 0.007*"news" + 0.007*"read" + 0.007*"candidates" + 0.007*"city" + 0.007*"#comelec"'),
 (4,
  '0.028*"balota" + 0.024*"garcia" + 0.024*"comelec" + 0.023*"george" + 0.019*"chairman" + 0.013*"media" + 0.009*"watch" + 0.009*"mendoza" + 0.009*"client" + 0.008*"pangalan"'),
 (5,
  '0.019*"duterte" + 0.014*"marcos" + 0.011*"senatorial" + 0.010*"bbm" + 0.009*"dds" + 0.009*"bato" + 0.009*"ab

In [62]:
import pyLDAvis, pyLDAvis.gensim_models

pyLDAvis.enable_notebook()

vis = pyLDAvis.gensim_models.prepare(lda, corpus, dictionary)
pyLDAvis.save_html(vis, "results3.html")